In [15]:
%pip install qiskit==1.2.4
%pip install qiskit-aer==0.15.1
%pip install pylatexenc==2.10

from qiskit import QuantumCircuit
from qiskit.converters import circuit_to_gate
from qiskit.visualization import array_to_latex
from qiskit.quantum_info import Operator
from qiskit.quantum_info import Statevector
from qiskit import transpile
from qiskit.providers.basic_provider import BasicSimulator
from qiskit.visualization import plot_histogram
from qiskit.circuit import ControlledGate
from qiskit import QuantumRegister, ClassicalRegister
from qiskit import QuantumCircuit, transpile
import math

In [16]:
simulator = BasicSimulator()

def get_quantum_random_bit():
    qrng_qc = QuantumCircuit(1, 1)
    qrng_qc.h(0)
    qrng_qc.measure(0, 0)
    job = transpile(qrng_qc, simulator)
    result = simulator.run(job, shots=1).result()
    counts = result.get_counts()
    return int(list(counts.keys())[0])

NUM_QUBITS = 24 # Using 24 due to simulator memory limits
ERROR_THRESHOLD = 0.10

q_reg = QuantumRegister(NUM_QUBITS, 'q')
eve_c_reg = ClassicalRegister(NUM_QUBITS, 'eve_c')
bob_c_reg = ClassicalRegister(NUM_QUBITS, 'bob_c')
qc = QuantumCircuit(q_reg, eve_c_reg, bob_c_reg)

In [17]:
alice_bits = [get_quantum_random_bit() for _ in range(NUM_QUBITS)]
alice_bases = [get_quantum_random_bit() for _ in range(NUM_QUBITS)]

for i in range(NUM_QUBITS):
    if alice_bits[i] == 1:
        qc.x(i)
    if alice_bases[i] == 1:
        qc.h(i)

In [18]:
eve_bases = [get_quantum_random_bit() for _ in range(NUM_QUBITS)]

for i in range(NUM_QUBITS):
    if eve_bases[i] == 1:
        qc.h(i)

    # Intercept and measure
    qc.measure(q_reg[i], eve_c_reg[i])

    # Prep state to send forward to Bob
    if eve_bases[i] == 1:
        qc.h(i)

In [19]:
bob_bases = [get_quantum_random_bit() for _ in range(NUM_QUBITS)]

for i in range(NUM_QUBITS):
    if bob_bases[i] == 1:
        qc.h(i)
    qc.measure(q_reg[i], bob_c_reg[i])

job = transpile(qc, simulator)
result = simulator.run(job, shots=1).result()
counts = result.get_counts()
measured_str = list(counts.keys())[0]

# Split the results
bob_str, eve_str = measured_str.split(' ')
bob_bits = [int(bit) for bit in bob_str[::-1]]
eve_bits = [int(bit) for bit in eve_str[::-1]]

In [20]:
alice_key = []
bob_key = []

# Sift keys
for i in range(NUM_QUBITS):
    if alice_bases[i] == bob_bases[i]:
        alice_key.append(alice_bits[i])
        bob_key.append(bob_bits[i])

# Sacrifice half the key to check for errors
sample_size = len(alice_key) // 2
alice_sample = alice_key[:sample_size]
bob_sample = bob_key[:sample_size]

errors = sum(1 for i in range(sample_size) if alice_sample[i] != bob_sample[i])
error_rate = errors / sample_size if sample_size > 0 else 0

print(f"Sifted key length: {len(alice_key)}")
print(f"Sample size used:  {sample_size}")
print(f"Errors found:      {errors}")
print(f"Error rate:        {error_rate*100:.2f}%")
print("-" * 30)

if error_rate > ERROR_THRESHOLD:
    print("ALERT: Attack detected! Key has been compromised and will be discarded.")
else:
    print("Key is safe [or attacker (Eve) got lucky with a small sample].")

Sifted key length: 16
Sample size used:  8
Errors found:      3
Error rate:        37.50%
------------------------------
ALERT: Attack detected! Key has been compromised and will be discarded.


In [21]:
# Another kind of attack: Entanglement attack

q_bonus = QuantumRegister(12, 'ab_q')
eve_q = QuantumRegister(12, 'eve_ancilla')
eve_c = ClassicalRegister(12, 'eve_c')
bob_c = ClassicalRegister(12, 'bob_c')

qc_bonus = QuantumCircuit(q_bonus, eve_q, eve_c, bob_c)

# Alice
alice_bits_b = [get_quantum_random_bit() for _ in range(12)]
alice_bases_b = [get_quantum_random_bit() for _ in range(12)]

for i in range(12):
    if alice_bits_b[i] == 1: qc_bonus.x(i)
    if alice_bases_b[i] == 1: qc_bonus.h(i)

# Eve uses a CNOT to entangle instead of measuring directly
for i in range(12):
    qc_bonus.cx(q_bonus[i], eve_q[i])
    qc_bonus.measure(eve_q[i], eve_c[i])

# Bob
bob_bases_b = [get_quantum_random_bit() for _ in range(12)]
for i in range(12):
    if bob_bases_b[i] == 1: qc_bonus.h(i)
    qc_bonus.measure(q_bonus[i], bob_c[i])

job_b = transpile(qc_bonus, simulator)
result_b = simulator.run(job_b, shots=1).result()
counts_b = result_b.get_counts()
bob_str_b, eve_str_b = list(counts_b.keys())[0].split(' ')
bob_bits_b = [int(bit) for bit in bob_str_b[::-1]]

# Detect
alice_key_b = []
bob_key_b = []
for i in range(12):
    if alice_bases_b[i] == bob_bases_b[i]:
        alice_key_b.append(alice_bits_b[i])
        bob_key_b.append(bob_bits_b[i])

sample_size_b = len(alice_key_b) // 2
errors_b = sum(1 for i in range(sample_size_b) if alice_key_b[i] != bob_key_b[i])
error_rate_b = errors_b / sample_size_b if sample_size_b > 0 else 0

print(f"\n--- Entanglement Attack ---")
print(f"Sifted key length: {len(alice_key_b)}")
print(f"Sample size: {sample_size_b}")
print(f"Errors found: {errors_b}")
print(f"Error rate: {error_rate_b*100:.2f}%")

if error_rate_b > 0.10:
    print("Attack detected! Error rate is over 10% threshold.")
else:
    print("Key is safe [or attacker (Eve) got lucky with a small sample].")


--- Entanglement Attack ---
Sifted key length: 7
Sample size: 3
Errors found: 1
Error rate: 33.33%
Attack detected! Error rate is over 10% threshold.
